# Forest Sketch baseline comparison

This notebook compares the five planned baselines from the README with the full Forest Sketch representation:

1. Original features
2. Initial random projection
3. One-shot tree-path projection
4. Iterative forest sketch without concatenation
5. Matched random sparse-feature control
6. Full Forest Sketch

The first experiment uses a binary classification dataset and a common logistic-regression downstream model. Replace the dataset and metric for other tasks.

In [1]:
import numpy as np
from scipy import sparse
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.utils.validation import check_array, check_is_fitted

from forestsketch import DecisionPathEncoder, ForestSketchEstimator, make_projector
RANDOM_STATE = 42
N_COMPONENTS = 16
N_ITERATIONS = 2
    

In [2]:
dataset = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(
    dataset.data,
    dataset.target,
    test_size=0.25,
    stratify=dataset.target,
    random_state=RANDOM_STATE,
)

def forest_template(seed=RANDOM_STATE):
    return RandomForestClassifier(
        n_estimators=80,
        max_depth=10,
        random_state=seed,
        n_jobs=-1,
    )

def downstream():
    return make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=3000, random_state=RANDOM_STATE),
    )


In [3]:
class OneShotTreePathTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, estimator, n_components=32, random_state=None):
        self.estimator = estimator
        self.n_components = n_components
        self.random_state = random_state

    def fit(self, X, y):
        X = check_array(X, dtype=np.float64, ensure_2d=True)
        self.n_features_in_ = X.shape[1]
        self.forest_ = clone(self.estimator).fit(X, y)
        self.encoder_ = DecisionPathEncoder().fit(self.forest_)
        V = self.encoder_.transform(X)
        self.projector_ = make_projector(
            self.n_components,
            'gaussian',
            self.random_state,
        ).fit(V)
        return self

    def transform(self, X):
        check_is_fitted(self, ('forest_', 'encoder_', 'projector_'))
        X = check_array(X, dtype=np.float64, ensure_2d=True)
        if X.shape[1] != self.n_features_in_:
            raise ValueError('X has an unexpected number of features')
        return self.projector_.transform(self.encoder_.transform(X))


In [4]:
class NoConcatForestSketch(BaseEstimator, TransformerMixin):
    def __init__(self, estimator, n_components=32, n_iterations=1, random_state=None):
        self.estimator = estimator
        self.n_components = n_components
        self.n_iterations = n_iterations
        self.random_state = random_state

    def fit(self, X, y):
        X = check_array(X, dtype=np.float64, ensure_2d=True)
        self.n_features_in_ = X.shape[1]
        seeds = np.random.RandomState(self.random_state).randint(
            0, np.iinfo(np.int32).max, size=self.n_iterations + 1
        )
        self.initial_projector_ = make_projector(
            self.n_components, 'sparse', seeds[0]
        ).fit(X)
        current = self.initial_projector_.transform(X)
        self.forests_ = []
        self.encoders_ = []
        self.projectors_ = []

        for iteration in range(self.n_iterations):
            forest = clone(self.estimator).fit(current, y)
            encoder = DecisionPathEncoder().fit(forest)
            V = encoder.transform(current)
            projector = make_projector(
                self.n_components, 'gaussian', seeds[iteration + 1]
            ).fit(V)
            current = projector.transform(V)
            self.forests_.append(forest)
            self.encoders_.append(encoder)
            self.projectors_.append(projector)
        return self

    def transform(self, X):
        check_is_fitted(self, ('initial_projector_', 'forests_'))
        X = check_array(X, dtype=np.float64, ensure_2d=True)
        current = self.initial_projector_.transform(X)
        for encoder, projector in zip(self.encoders_, self.projectors_):
            current = projector.transform(encoder.transform(current))
        return current


In [5]:
class MatchedRandomSparsePathTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, estimator, n_components=32, random_state=None):
        self.estimator = estimator
        self.n_components = n_components
        self.random_state = random_state

    def fit(self, X, y):
        X = check_array(X, dtype=np.float64, ensure_2d=True)
        self.n_features_in_ = X.shape[1]
        self.forest_ = clone(self.estimator).fit(X, y)
        self.encoder_ = DecisionPathEncoder().fit(self.forest_)
        V = self.encoder_.transform(X)
        self.n_path_features_ = V.shape[1]
        self.nnz_per_row_ = max(1, int(round(V.nnz / V.shape[0])))
        random_V = self._random_sparse_features(V.shape[0])
        self.projector_ = make_projector(
            self.n_components, 'gaussian', self.random_state
        ).fit(random_V)
        return self

    def _random_sparse_features(self, n_rows):
        rng = np.random.RandomState(self.random_state)
        rows = np.repeat(np.arange(n_rows), self.nnz_per_row_)
        cols = rng.randint(0, self.n_path_features_, size=rows.size)
        signs = rng.choice(np.array([-1.0, 1.0]), size=rows.size)
        return sparse.coo_matrix(
            (signs, (rows, cols)),
            shape=(n_rows, self.n_path_features_),
        ).tocsr()

    def transform(self, X):
        check_is_fitted(self, ('forest_', 'encoder_', 'projector_'))
        X = check_array(X, dtype=np.float64, ensure_2d=True)
        random_V = self._random_sparse_features(X.shape[0])
        return self.projector_.transform(random_V)


In [6]:
models = {
    'original features': downstream(),
    'initial random projection': make_pipeline(
        ForestSketchEstimator(
            estimator=forest_template(),
            n_components=N_COMPONENTS,
            n_iterations=0,
            random_state=RANDOM_STATE,
        ),
        downstream(),
    ),
    'one-shot tree-path projection': make_pipeline(
        OneShotTreePathTransformer(
            estimator=forest_template(),
            n_components=N_COMPONENTS,
            random_state=RANDOM_STATE,
        ),
        downstream(),
    ),
    'iterative without concatenation': make_pipeline(
        NoConcatForestSketch(
            estimator=forest_template(),
            n_components=N_COMPONENTS,
            n_iterations=N_ITERATIONS,
            random_state=RANDOM_STATE,
        ),
        downstream(),
    ),
    'matched random sparse control': make_pipeline(
        MatchedRandomSparsePathTransformer(
            estimator=forest_template(),
            n_components=N_COMPONENTS,
            random_state=RANDOM_STATE,
        ),
        downstream(),
    ),
    'full Forest Sketch': make_pipeline(
        ForestSketchEstimator(
            estimator=forest_template(),
            n_components=N_COMPONENTS,
            n_iterations=N_ITERATIONS,
            random_state=RANDOM_STATE,
            normalization='none',
        ),
        downstream(),
    ),
}


In [7]:
results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    prediction = model.predict(X_test)
    results.append((name, accuracy_score(y_test, prediction)))

for name, score in sorted(results, key=lambda item: item[1], reverse=True):
    print(f'{name:40s} accuracy={score:.3f}')


original features                        accuracy=0.986
initial random projection                accuracy=0.972
one-shot tree-path projection            accuracy=0.944
full Forest Sketch                       accuracy=0.944
iterative without concatenation          accuracy=0.916
matched random sparse control            accuracy=0.622


Interpret the result together with fit time, transform time, memory, and model size. A single accuracy win is not sufficient evidence: repeat this comparison across seeds and datasets, then use cross-validation to select n_components and n_iterations.